# Chapter 4: Markov Chain Monte Carlo (MCMC)

When the posterior cannot be computed analytically, we sample from it.
MCMC generates a sequence of samples whose long-run distribution equals the posterior —
without ever computing the normalisation constant P(D).

This notebook implements Metropolis-Hastings from scratch, then uses `emcee` (the
standard MCMC sampler in astrophysics) for a realistic fitting problem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)
print("Ready.")


## 4.1 Metropolis-Hastings from Scratch

The algorithm:
1. Start at θ⁽⁰⁾
2. Propose θ* ~ q(θ*|θ⁽ᵗ⁾)  (e.g. Gaussian random walk)
3. Compute acceptance ratio: r = π(θ*)/π(θ⁽ᵗ⁾) × q(θ⁽ᵗ⁾|θ*)/q(θ*|θ⁽ᵗ⁾)
4. Accept θ* with probability min(1, r); otherwise stay at θ⁽ᵗ⁾

For symmetric proposals: r = π(θ*)/π(θ⁽ᵗ⁾)


In [ ]:
def metropolis_hastings(log_target, theta0, n_steps, proposal_std):
    # Simple Metropolis sampler with Gaussian random-walk proposal.
    
    Parameters
    ----------
    log_target    : callable, log of unnormalised target density
    theta0        : array, starting position
    n_steps       : int, number of MCMC steps
    proposal_std  : float or array, std of Gaussian proposal
    
    Returns
    -------
    chain         : array (n_steps+1, d)
    acceptance    : float, acceptance rate
    d = len(np.atleast_1d(theta0))
    chain = np.zeros((n_steps + 1, d))
    chain[0] = theta0
    n_accept = 0
    
    for t in range(n_steps):
        current   = chain[t]
        proposal  = current + proposal_std * np.random.randn(d)
        
        log_r = log_target(proposal) - log_target(current)
        
        if np.log(np.random.rand()) < log_r:
            chain[t+1] = proposal
            n_accept  += 1
        else:
            chain[t+1] = current
    
    return chain, n_accept / n_steps


# Target: bivariate Gaussian with correlation (easy known test case)
mean_target = np.array([2.0, 3.0])
cov_target  = np.array([[1.0, 0.7], [0.7, 1.5]])
rv_target   = stats.multivariate_normal(mean_target, cov_target)
log_target  = lambda x: rv_target.logpdf(x)

# Run with 3 different proposal widths
proposal_stds = [0.05, 0.5, 5.0]
n_steps = 5000
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, pstd in enumerate(proposal_stds):
    chain, acc = metropolis_hastings(log_target, np.array([0., 0.]), n_steps, pstd)
    
    # Trace plot (first 500 steps)
    axes[0, col].plot(chain[:500, 0], alpha=0.8, color="#3B8BD4", lw=0.8)
    axes[0, col].axhline(mean_target[0], color="#D85A30", lw=1.5, ls="--")
    axes[0, col].set_title(f"Proposal σ={pstd}  |  Acceptance={acc:.1%}")
    axes[0, col].set_xlabel("Step"); axes[0, col].set_ylabel("θ₁")
    
    # Scatter of samples vs true distribution
    burn = 500
    axes[1, col].scatter(chain[burn:, 0], chain[burn:, 1],
                         alpha=0.15, s=3, color="#3B8BD4")
    # True density contours
    x_g = np.linspace(-2, 6, 100)
    y_g = np.linspace(-1, 7, 100)
    X, Y = np.meshgrid(x_g, y_g)
    pos  = np.dstack((X, Y))
    axes[1, col].contour(X, Y, rv_target.pdf(pos), levels=5,
                         colors=["#D85A30"], linewidths=1.5)
    axes[1, col].set_xlabel("θ₁"); axes[1, col].set_ylabel("θ₂")
    axes[1, col].set_title(f"Samples (burn={burn}) vs True contours")

plt.suptitle("Metropolis-Hastings: Effect of Proposal Width", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Optimal acceptance rate for 2D Gaussian ≈ 23.4%")
print(f"  σ=0.05: {acc:.1%} → too small (very correlated chain, slow mixing)")


## 4.2 MCMC Diagnostics: Convergence and Effective Sample Size

How do we know when the chain has converged?

**R̂ (Gelman-Rubin):** Run multiple chains from different starting points.
R̂ → 1 when chains are sampling the same distribution. R̂ < 1.01 = good.

**ESS (Effective Sample Size):** Due to autocorrelation, N raw samples ≠ N independent samples.
ESS = N / τ_int where τ_int is the integrated autocorrelation time.


In [ ]:
def autocorrelation(chain, max_lag=100):
    # Compute normalised autocorrelation function
    n = len(chain)
    c = chain - chain.mean()
    acf = np.correlate(c, c, mode='full')[n-1:]
    return acf[:max_lag+1] / acf[0]

def integrated_autocorr(chain, c=5):
    # Estimate integrated autocorrelation time using Sokal windowed estimator
    acf = autocorrelation(chain, max_lag=len(chain)//2)
    tau = 1.0
    for k in range(1, len(acf)):
        tau += 2.0 * acf[k]
        if k >= c * tau:   # window condition
            break
    return tau

def gelman_rubin(chains):
    # Compute R-hat for an array of chains, shape (M, N).
    M, N    = chains.shape
    chain_means = chains.mean(axis=1)
    grand_mean  = chain_means.mean()
    B = N / (M-1) * np.sum((chain_means - grand_mean)**2)
    W = chains.var(axis=1, ddof=1).mean()
    V_hat = (N-1)/N * W + (M+1)/(M*N) * B
    return np.sqrt(V_hat / W)

# Run 4 chains from different starting points
n_chains = 4
n_steps  = 4000
starts   = [np.array([5., 7.]), np.array([-1., -2.]),
            np.array([8., 0.]), np.array([0.,  8.])]
chains   = []

for s in starts:
    c, _ = metropolis_hastings(log_target, s, n_steps, proposal_std=0.8)
    chains.append(c)

chains_arr = np.array([c[:, 0] for c in chains])   # shape (4, N)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Trace plots
for i, (c, start) in enumerate(zip(chains, starts)):
    axes[0].plot(c[:, 0], alpha=0.7, lw=0.8, label=f"Chain {i+1}")
axes[0].axhline(mean_target[0], color="black", lw=2, ls="--", label="True mean")
axes[0].set_xlabel("Step"); axes[0].set_ylabel("θ₁")
axes[0].set_title("Trace Plots — 4 Chains from Overdispersed Starts")
axes[0].legend(fontsize=8, frameon=False)

# Gelman-Rubin as function of steps
step_checkpoints = np.geomspace(10, n_steps, 30).astype(int)
Rhat_vals = [gelman_rubin(chains_arr[:, :s]) for s in step_checkpoints]
axes[1].semilogx(step_checkpoints, Rhat_vals, "#3B8BD4", lw=2)
axes[1].axhline(1.01, color="#D85A30", ls="--", lw=2, label="R̂ = 1.01 threshold")
axes[1].set_xlabel("Steps per chain"); axes[1].set_ylabel("R̂")
axes[1].set_title("Gelman-Rubin Convergence Statistic")
axes[1].legend(frameon=False)

# Autocorrelation function + ESS
chain_after_burnin = chains[0][500:, 0]
acf_vals = autocorrelation(chain_after_burnin, max_lag=80)
tau_int  = integrated_autocorr(chain_after_burnin)
ess      = len(chain_after_burnin) / tau_int

axes[2].plot(acf_vals, "#1D9E75", lw=2)
axes[2].axhline(0, color="black", lw=1, ls="--")
axes[2].set_xlabel("Lag k"); axes[2].set_ylabel("Autocorrelation ρ(k)")
axes[2].set_title(f"Autocorrelation Function
τ_int = {tau_int:.1f},  ESS = {ess:.0f}")

plt.tight_layout()
plt.show()

print(f"R̂ (after {n_steps} steps): {Rhat_vals[-1]:.4f}  (< 1.01 = converged)")
print(f"Integrated autocorrelation time: {tau_int:.1f}")
print(f"Raw samples: {len(chain_after_burnin)},  Effective samples: {ess:.0f}")


## 4.3 emcee — Ensemble Sampler for Astrophysics

`emcee` (Foreman-Mackey et al. 2013) is the most widely used MCMC sampler
in astrophysics. It uses an **affine-invariant ensemble** of walkers with the
**stretch move** — no proposal covariance tuning required.


In [ ]:
try:
    import emcee
    import corner
    HAS_EMCEE = True
    print(f"emcee version: {emcee.__version__}")
except ImportError:
    HAS_EMCEE = False
    print("emcee not installed. Run: pip install emcee corner")
    print("Showing synthetic results instead.")


In [ ]:
# Line fitting with emcee: y = m*x + b + intrinsic scatter
# This is the standard astrophysical use case

np.random.seed(1)
m_true, b_true, sigma_int_true = 2.3, 0.5, 1.2

x_data = np.sort(np.random.uniform(0, 10, 40))
y_data = m_true*x_data + b_true + sigma_int_true*np.random.randn(40)
y_err  = 0.5 + 0.3*np.random.rand(40)   # heteroscedastic errors
y_data += y_err * np.random.randn(40)

def log_prior(theta):
    m, b, log_sigma = theta
    if not (-10 < m < 10):    return -np.inf
    if not (-20 < b < 20):    return -np.inf
    if not (-3  < log_sigma < 3): return -np.inf
    return 0.0   # flat prior within bounds

def log_likelihood(theta, x, y, yerr):
    m, b, log_sigma = theta
    sigma_tot = np.sqrt(yerr**2 + np.exp(2*log_sigma))
    model     = m*x + b
    return -0.5 * np.sum(((y - model)/sigma_tot)**2 + np.log(2*np.pi*sigma_tot**2))

def log_posterior(theta, x, y, yerr):
    lp = log_prior(theta)
    return lp + log_likelihood(theta, x, y, yerr) if np.isfinite(lp) else -np.inf

if HAS_EMCEE:
    ndim, nwalkers = 3, 32
    theta_init = np.array([2.0, 0.3, 0.1])
    p0 = theta_init + 0.1*np.random.randn(nwalkers, ndim)
    
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior,
                                     args=(x_data, y_data, y_err))
    print("Burn-in (500 steps)...")
    sampler.run_mcmc(p0, 500, progress=True)
    sampler.reset()
    print("Production (3000 steps)...")
    sampler.run_mcmc(None, 3000, progress=True)
    
    flat_samples = sampler.get_chain(flat=True)
    
    # Convergence
    tau  = sampler.get_autocorr_time(quiet=True)
    Rhat_vals_emcee = []
    for i in range(ndim):
        chains_i = sampler.get_chain()[:, :, i].T   # (nwalkers, nsteps)
        Rhat_vals_emcee.append(gelman_rubin(chains_i))
    
    print(f"
Autocorrelation times: {np.round(tau, 1)}")
    print(f"R̂ values: {[f'{r:.4f}' for r in Rhat_vals_emcee]}")
    print(f"Total samples: {len(flat_samples)},  Samples/autocorr: {len(flat_samples)/tau.mean():.0f}")
else:
    # Generate fake samples for demonstration
    m_samples   = np.random.normal(m_true, 0.08, 5000)
    b_samples   = np.random.normal(b_true, 0.4,  5000)
    ls_samples  = np.random.normal(np.log(sigma_int_true), 0.12, 5000)
    flat_samples = np.column_stack([m_samples, b_samples, ls_samples])
    print("Using synthetic posterior samples for demonstration.")


In [ ]:
# Plot: posterior corner + best-fit line
param_names = ["slope m", "intercept b", "ln(σ_int)"]
truths      = [m_true, b_true, np.log(sigma_int_true)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Data + best-fit model
x_plot  = np.linspace(0, 10, 200)
m_med   = np.median(flat_samples[:, 0])
b_med   = np.median(flat_samples[:, 1])

# Sample 200 posterior draws for uncertainty band
idx = np.random.choice(len(flat_samples), 200, replace=False)
for i in idx:
    m_s, b_s, _ = flat_samples[i]
    axes[0].plot(x_plot, m_s*x_plot + b_s, color="#3B8BD4", alpha=0.05, lw=1)

axes[0].errorbar(x_data, y_data, yerr=y_err, fmt='o', color="#888780",
                 ms=4, elinewidth=0.8, label="Data")
axes[0].plot(x_plot, m_med*x_plot + b_med, "#D85A30", lw=2.5,
             label=f"Posterior median: m={m_med:.2f}, b={b_med:.2f}")
axes[0].plot(x_plot, m_true*x_plot + b_true, "#1D9E75", lw=2, ls="--",
             label=f"True: m={m_true}, b={b_true}")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title("Line Fit with Posterior Uncertainty")
axes[0].legend(fontsize=9, frameon=False)

# 1D marginal posteriors
for i, (name, truth) in enumerate(zip(param_names, truths)):
    c = ["#3B8BD4","#1D9E75","#D85A30"][i]
    q = np.percentile(flat_samples[:,i], [5,16,50,84,95])
    offset = i*0.35
    axes[1].fill_betweenx([offset, offset+0.3],
                          q[1], q[3], color=c, alpha=0.5)
    axes[1].plot([q[2], q[2]], [offset, offset+0.3], color=c, lw=2.5)
    axes[1].plot([q[0], q[4]], [offset+0.15, offset+0.15],
                 color=c, lw=1, alpha=0.5)
    axes[1].axvline(truth, color=c, lw=1.5, ls="--", alpha=0.7)
    axes[1].text(q[4]+0.05, offset+0.15, f"{name}
={q[2]:.3f}+{q[3]-q[2]:.3f}-{q[2]-q[1]:.3f}",
                 va='center', fontsize=9, color=c)

axes[1].set_title("Posterior Marginals (median + 68%/90% CI)")
axes[1].set_yticks([]); axes[1].set_xlabel("Parameter value")

plt.tight_layout()
plt.show()


In [ ]:
if HAS_EMCEE:
    try:
        fig = corner.corner(flat_samples,
                            labels=param_names,
                            truths=truths,
                            truth_color="#D85A30",
                            quantiles=[0.16, 0.5, 0.84],
                            show_titles=True,
                            title_fmt=".3f",
                            plot_datapoints=False,
                            fill_contours=True)
        plt.suptitle("Corner Plot — Joint and Marginal Posteriors", y=1.01, fontsize=12)
        plt.show()
    except Exception as e:
        print(f"Corner plot skipped: {e}")


## 4.4 Summary

| Algorithm | Proposal | Acceptance | Best for |
|-----------|----------|------------|----------|
| Metropolis | Gaussian random walk | 23% optimal | Low dim, simple posterior |
| emcee | Stretch move (ensemble) | auto-adapts | d < 20, standard astro use |
| HMC/NUTS | Gradient-based leapfrog | ~65–90% | d > 10, differentiable models |
| Gibbs | Full conditionals | 100% | Conjugate hierarchical models |

**Convergence checklist:** R̂ < 1.01, ESS > 200 per parameter, trace plots look like fuzzy caterpillars.

Next: Chapter 5 — Nested Sampling, which also computes the model evidence Z.


In [ ]:
print("Chapter 4 complete!")
print("The emcee pattern to remember:")
lines = [
    "  def log_posterior(theta, data):",
    "      lp = log_prior(theta)",
    "      if not np.isfinite(lp): return -np.inf",
    "      return lp + log_likelihood(theta, data)",
    "  sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior, args=[data])",
    "  sampler.run_mcmc(p0, n_burn);  sampler.reset()",
    "  sampler.run_mcmc(None, n_prod)",
    "  flat = sampler.get_chain(flat=True)",
]
print('\n'.join(lines))
